# 07 — Paper-style TimeVAE Train + Generate

This notebook trains a stronger **paper-style TimeVAE baseline** for the IR lab project.

The key change from the previous weak version is that TimeVAE is trained internally on one fused multivariate sequence:

```text
Fused TimeVAE input: [N, 512, 6]
Channels: ACC_x, ACC_y, ACC_z, BVP, EDA, TEMP
```

Then generated fused samples are converted back to the same native-rate format used by KoVAE:

```text
ACC  -> [N, 256, 3]
BVP  -> [N, 512, 1]
SLOW -> [N,  32, 2]
```

So the comparison, realism/diversity, and downstream evaluation notebooks can use the exact same saved file names.

Output method:

```text
timevae/prior_v1
```


In [1]:

# ============================================================
# 07_train_generate_timevae_paperstyle.py
#
# Paper-style TimeVAE baseline adapted to native-rate PPG-DaLiA.
#
# Internal model input:
#   fused [N, 512, 6]
#   channels = ACC_x, ACC_y, ACC_z, BVP, EDA, TEMP
#
# Saved synthetic output, compatible with KoVAE notebooks:
#   data/synthetic_subjects/timevae/prior_v1/generated_subjects_X_acc_32hz.npy
#   data/synthetic_subjects/timevae/prior_v1/generated_subjects_X_bvp_64hz.npy
#   data/synthetic_subjects/timevae/prior_v1/generated_subjects_X_slow_4hz.npy
#   data/synthetic_subjects/timevae/prior_v1/generated_subjects_all_y.npy
#   data/synthetic_subjects/timevae/prior_v1/generated_subjects_all_subject.npy
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
import random
import time
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Configuration
# ============================================================

TIMEVAE_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "model_family": "timevae",
    "generation_method": "prior_v1",
    "method_display_name": "TimeVAE-Prior",

    "real_dir": "data/processed/native_rates",

    "configs_dir": "configs",
    "models_checkpoint_dir": "models/checkpoints",
    "models_generator_dir": "models/generators",
    "results_dir": "results/timevae",
    "figures_dir": "figures/timevae",
    "logs_dir": "logs",

    "synthetic_base_dir": "data/synthetic_subjects/timevae",
    "generation_results_base_dir": "results/timevae_generation",
    "generation_figures_base_dir": "figures/timevae_generation",

    "train_subjects": ["S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"],
    "val_subjects": ["S14", "S15"],
    "test_subjects": ["S7", "S8", "S10"],

    "activity_ids": [1, 2, 3, 4, 5, 6, 7, 8],

    "random_seed": 42,

    # Training
    "batch_size": 128,
    "num_workers": 0,
    "max_epochs": 100,
    "patience": 15,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "gradient_clip_norm": 5.0,

    # Paper-style fused TimeVAE architecture
    "input_length": 512,
    "input_channels": 6,
    "latent_dim": 128,
    "hidden_dim": 128,
    "activity_embedding_dim": 16,
    "dropout": 0.10,

    # VAE objective. Higher reconstruction_weight + lower beta_kl helps avoid flat reconstructions.
    "reconstruction_weight": 3.5,
    "beta_kl": 0.0001,
    "kl_warmup_epochs": 30,
    "activity_loss_weight": 0.10,

    "use_amp": True,

    # Generation
    "num_synthetic_subjects": 10,
    "windows_per_synthetic_subject": 3000,
    "generation_batch_size": 512,
    "activity_sampling_mode": "train_distribution",
    "clip_generated_to_train_range": False,

    # Plots
    "save_reconstruction_examples": True,
    "num_reconstruction_examples": 4,
}

RAW_ARRAY_CONFIGS = {
    "acc": {
        "filename": "all_X_acc_32hz.npy",
        "shape_tail": (256, 3),
    },
    "bvp": {
        "filename": "all_X_bvp_64hz.npy",
        "shape_tail": (512, 1),
    },
    "slow": {
        "filename": "all_X_slow_4hz.npy",
        "shape_tail": (32, 2),
    },
}

FUSED_CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]


# ============================================================
# Utilities
# ============================================================

def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def require_file(path: Path) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def subject_sort_key(value):
    text = str(value)
    if text.startswith("S") and text[1:].isdigit():
        return (0, int(text[1:]))
    digits = "".join(ch for ch in text if ch.isdigit())
    if digits:
        return (1, int(digits), text)
    return (2, text)


def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("timevae_paperstyle")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    log_path.parent.mkdir(parents=True, exist_ok=True)

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(formatter)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)
    return logger


def get_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    method = str(config["generation_method"])
    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "configs_dir": root / config["configs_dir"],
        "checkpoint_dir": root / config["models_checkpoint_dir"],
        "generator_dir": root / config["models_generator_dir"],
        "results_dir": root / config["results_dir"],
        "figures_dir": root / config["figures_dir"],
        "logs_dir": root / config["logs_dir"],
        "synthetic_method_dir": root / config["synthetic_base_dir"] / method,
        "generation_results_dir": root / config["generation_results_base_dir"] / method,
        "generation_figures_dir": root / config["generation_figures_base_dir"] / method,
        "generation_results_base_dir": root / config["generation_results_base_dir"],
    }


def count_parameters(model: nn.Module) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_parameters": int(total), "trainable_parameters": int(trainable)}


def to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def safe_numpy(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().numpy()


# ============================================================
# Native-rate loading and fused representation
# ============================================================

def check_array_shape(name: str, arr: np.ndarray, expected_tail: Tuple[int, int]) -> None:
    if arr.ndim != 3 or tuple(arr.shape[1:]) != tuple(expected_tail):
        raise ValueError(f"{name}: expected [N,{expected_tail[0]},{expected_tail[1]}], got {arr.shape}")


def load_native_arrays(real_dir: Path, config: Dict) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray]:
    X = {}
    for key, arr_cfg in RAW_ARRAY_CONFIGS.items():
        path = require_file(real_dir / arr_cfg["filename"])
        arr = np.load(path).astype(np.float32)
        check_array_shape(key, arr, arr_cfg["shape_tail"])
        X[key] = arr

    y = np.load(require_file(real_dir / "all_y.npy")).astype(np.int64)
    subjects = np.load(require_file(real_dir / "all_subject.npy"), allow_pickle=True).astype(str)

    if len(y) != len(subjects):
        raise ValueError(f"y/subject mismatch: {len(y)} vs {len(subjects)}")

    for key, arr in X.items():
        if len(arr) != len(y):
            raise ValueError(f"{key}/y mismatch: {len(arr)} vs {len(y)}")

    keep = np.isin(y, np.asarray(config["activity_ids"], dtype=np.int64))
    X = {key: value[keep] for key, value in X.items()}
    y = y[keep]
    subjects = subjects[keep]
    return X, y, subjects


def resample_time_axis(X: np.ndarray, target_len: int) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    if X.ndim != 3:
        raise ValueError(f"Expected [N,T,C], got {X.shape}")

    n, old_len, channels = X.shape
    if old_len == target_len:
        return X.copy().astype(np.float32)

    positions = np.linspace(0.0, old_len - 1, int(target_len), dtype=np.float32)
    left = np.floor(positions).astype(np.int64)
    right = np.minimum(left + 1, old_len - 1)
    weight = (positions - left).astype(np.float32)

    out = (1.0 - weight)[None, :, None] * X[:, left, :] + weight[None, :, None] * X[:, right, :]
    return out.astype(np.float32)


def native_to_fused512(X_native: Dict[str, np.ndarray]) -> np.ndarray:
    acc_512 = resample_time_axis(X_native["acc"], 512)
    bvp_512 = resample_time_axis(X_native["bvp"], 512)
    slow_512 = resample_time_axis(X_native["slow"], 512)
    fused = np.concatenate([acc_512, bvp_512, slow_512], axis=2).astype(np.float32)
    if fused.shape[1:] != (512, 6):
        raise ValueError(f"Fused shape mismatch: {fused.shape}")
    return fused


def fused512_to_native(fused: np.ndarray) -> Dict[str, np.ndarray]:
    fused = np.asarray(fused, dtype=np.float32)
    if fused.ndim != 3 or fused.shape[1:] != (512, 6):
        raise ValueError(f"Expected fused [N,512,6], got {fused.shape}")

    acc_512 = fused[:, :, 0:3]
    bvp = fused[:, :, 3:4]
    slow_512 = fused[:, :, 4:6]

    acc = resample_time_axis(acc_512, 256)
    slow = resample_time_axis(slow_512, 32)

    return {
        "acc": acc.astype(np.float32),
        "bvp": bvp.astype(np.float32),
        "slow": slow.astype(np.float32),
    }


def validate_subject_split(subjects: np.ndarray, config: Dict) -> None:
    available = set(subjects.astype(str))
    missing_train = sorted(set(config["train_subjects"]) - available, key=subject_sort_key)
    missing_val = sorted(set(config["val_subjects"]) - available, key=subject_sort_key)
    missing_test = sorted(set(config["test_subjects"]) - available, key=subject_sort_key)
    if missing_train or missing_val or missing_test:
        raise ValueError(f"Missing subjects: train={missing_train}, val={missing_val}, test={missing_test}")


def make_split_indices(subjects: np.ndarray, config: Dict) -> Dict[str, np.ndarray]:
    subjects = subjects.astype(str)
    splits = {
        "train": set(config["train_subjects"]),
        "val": set(config["val_subjects"]),
        "test": set(config["test_subjects"]),
    }
    return {
        split_name: np.where(np.asarray([str(s) in selected for s in subjects], dtype=bool))[0]
        for split_name, selected in splits.items()
    }


def build_activity_mapping(activity_ids: List[int]) -> Dict:
    original_to_index = {str(int(label)): i for i, label in enumerate(activity_ids)}
    index_to_original = {str(i): int(label) for i, label in enumerate(activity_ids)}
    return {
        "activity_ids": [int(x) for x in activity_ids],
        "original_to_index": original_to_index,
        "index_to_original": index_to_original,
    }


def original_labels_to_indices(y: np.ndarray, mapping: Dict) -> np.ndarray:
    return np.asarray([mapping["original_to_index"][str(int(label))] for label in y], dtype=np.int64)


def indices_to_original_labels(y_idx: np.ndarray, mapping: Dict) -> np.ndarray:
    return np.asarray([mapping["index_to_original"][str(int(idx))] for idx in y_idx], dtype=np.int64)


def compute_fused_scaler_stats(X_fused: np.ndarray, train_idx: np.ndarray) -> Dict:
    train = X_fused[train_idx].astype(np.float32)
    mean = train.mean(axis=(0, 1), keepdims=True)
    std = train.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    min_value = train.min(axis=(0, 1), keepdims=True)
    max_value = train.max(axis=(0, 1), keepdims=True)
    return {
        "channel_names": FUSED_CHANNEL_NAMES,
        "mean": mean.reshape(-1).astype(float).tolist(),
        "std": std.reshape(-1).astype(float).tolist(),
        "min": min_value.reshape(-1).astype(float).tolist(),
        "max": max_value.reshape(-1).astype(float).tolist(),
    }


def normalize_fused(X: np.ndarray, stats: Dict) -> np.ndarray:
    mean = np.asarray(stats["mean"], dtype=np.float32).reshape(1, 1, -1)
    std = np.asarray(stats["std"], dtype=np.float32).reshape(1, 1, -1)
    return ((X.astype(np.float32) - mean) / std).astype(np.float32)


def inverse_normalize_fused(X: np.ndarray, stats: Dict) -> np.ndarray:
    mean = np.asarray(stats["mean"], dtype=np.float32).reshape(1, 1, -1)
    std = np.asarray(stats["std"], dtype=np.float32).reshape(1, 1, -1)
    return (X.astype(np.float32) * std + mean).astype(np.float32)


def clip_fused_to_train_range(X: np.ndarray, stats: Dict) -> np.ndarray:
    min_value = np.asarray(stats["min"], dtype=np.float32).reshape(1, 1, -1)
    max_value = np.asarray(stats["max"], dtype=np.float32).reshape(1, 1, -1)
    return np.clip(X, min_value, max_value).astype(np.float32)


class FusedTimeVAEDataset(Dataset):
    def __init__(self, X_fused_norm: np.ndarray, y_idx: np.ndarray, indices: np.ndarray):
        self.X = X_fused_norm.astype(np.float32)
        self.y_idx = y_idx.astype(np.int64)
        self.indices = indices.astype(np.int64)

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, item: int) -> Dict[str, torch.Tensor]:
        idx = int(self.indices[item])
        return {
            "x": torch.from_numpy(self.X[idx]),
            "activity": torch.tensor(self.y_idx[idx], dtype=torch.long),
            "global_index": torch.tensor(idx, dtype=torch.long),
        }


def make_dataloaders(X_fused_norm: np.ndarray, y_idx: np.ndarray, split_indices: Dict[str, np.ndarray], config: Dict) -> Tuple[DataLoader, DataLoader]:
    train_dataset = FusedTimeVAEDataset(X_fused_norm, y_idx, split_indices["train"])
    val_dataset = FusedTimeVAEDataset(X_fused_norm, y_idx, split_indices["val"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=True,
        num_workers=int(config["num_workers"]),
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=int(config["num_workers"]),
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    return train_loader, val_loader


def save_split_summary(y: np.ndarray, subjects: np.ndarray, split_indices: Dict[str, np.ndarray], paths: Dict[str, Path], config: Dict) -> pd.DataFrame:
    rows = []
    for split_name, indices in split_indices.items():
        row = {
            "split": split_name,
            "num_windows": int(len(indices)),
            "subjects": ",".join(sorted(np.unique(subjects[indices].astype(str)), key=subject_sort_key)),
        }
        for activity in config["activity_ids"]:
            row[f"activity_{activity}_windows"] = int(np.sum(y[indices] == int(activity)))
        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = paths["results_dir"] / "timevae_data_split_summary.csv"
    df.to_csv(out_path, index=False)
    return df


# ============================================================
# Paper-style fused TimeVAE model
# ============================================================

class PaperStyleTimeVAE(nn.Module):
    def __init__(
        self,
        input_channels: int,
        input_length: int,
        num_activities: int,
        latent_dim: int,
        hidden_dim: int,
        activity_embedding_dim: int,
        dropout: float,
    ):
        super().__init__()
        self.input_channels = int(input_channels)
        self.input_length = int(input_length)
        self.num_activities = int(num_activities)
        self.latent_dim = int(latent_dim)
        self.hidden_dim = int(hidden_dim)
        self.activity_embedding_dim = int(activity_embedding_dim)
        self.downsampled_length = self.input_length // 16

        if self.input_length % 16 != 0:
            raise ValueError("input_length must be divisible by 16 for this architecture.")

        self.activity_embedding = nn.Embedding(num_activities, activity_embedding_dim)

        self.encoder_conv = nn.Sequential(
            nn.Conv1d(input_channels, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
        )

        flat_dim = hidden_dim * self.downsampled_length
        encoder_dense_in = flat_dim + activity_embedding_dim

        self.encoder_dense = nn.Sequential(
            nn.Linear(encoder_dense_in, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
        )

        self.mu_head = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

        decoder_in = latent_dim + activity_embedding_dim
        self.decoder_dense = nn.Sequential(
            nn.Linear(decoder_in, hidden_dim * self.downsampled_length),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.decoder_deconv = nn.Sequential(
            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(hidden_dim, hidden_dim, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim, input_channels, kernel_size=1),
        )

        self.activity_classifier = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_activities),
        )

    def encode(self, x: torch.Tensor, activity: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: [B,T,C]
        h = x.transpose(1, 2)
        h = self.encoder_conv(h)
        h = h.flatten(start_dim=1)
        h_activity = self.activity_embedding(activity)
        h = torch.cat([h, h_activity], dim=1)
        h = self.encoder_dense(h)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h)
        logvar = torch.clamp(logvar, min=-8.0, max=8.0)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z: torch.Tensor, activity: torch.Tensor) -> torch.Tensor:
        h_activity = self.activity_embedding(activity)
        z_cond = torch.cat([z, h_activity], dim=1)
        h = self.decoder_dense(z_cond)
        h = h.view(z.shape[0], self.hidden_dim, self.downsampled_length)
        out = self.decoder_deconv(h)
        return out.transpose(1, 2)

    def forward(self, x: torch.Tensor, activity: torch.Tensor) -> Dict[str, torch.Tensor]:
        mu, logvar = self.encode(x, activity)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, activity)
        activity_logits = self.activity_classifier(mu)
        return {
            "recon": recon,
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "activity_logits": activity_logits,
        }


# ============================================================
# Training
# ============================================================

def compute_loss(batch: Dict[str, torch.Tensor], outputs: Dict[str, torch.Tensor], config: Dict, epoch: int) -> Dict[str, torch.Tensor]:
    recon_loss = F.mse_loss(outputs["recon"], batch["x"])

    mu = outputs["mu"]
    logvar = outputs["logvar"]
    kl_loss = -0.5 * torch.mean(torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp(), dim=1))

    activity_loss = F.cross_entropy(outputs["activity_logits"], batch["activity"])

    kl_warmup_epochs = max(1, int(config["kl_warmup_epochs"]))
    beta_kl = float(config["beta_kl"]) * min(1.0, float(epoch) / float(kl_warmup_epochs))

    total_loss = (
        float(config["reconstruction_weight"]) * recon_loss
        + beta_kl * kl_loss
        + float(config["activity_loss_weight"]) * activity_loss
    )

    with torch.no_grad():
        pred = outputs["activity_logits"].argmax(dim=1)
        activity_accuracy = (pred == batch["activity"]).float().mean()

    return {
        "total_loss": total_loss,
        "recon_loss": recon_loss.detach(),
        "kl_loss": kl_loss.detach(),
        "activity_loss": activity_loss.detach(),
        "activity_accuracy": activity_accuracy.detach(),
        "beta_kl_used": torch.tensor(beta_kl, device=total_loss.device),
    }


def aggregate(metric_sums: Dict[str, float], loss_dict: Dict[str, torch.Tensor], batch_size: int) -> None:
    for key, value in loss_dict.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + float(value.detach().cpu().item()) * int(batch_size)


def finalize(metric_sums: Dict[str, float], total_examples: int) -> Dict[str, float]:
    return {key: float(value / max(1, total_examples)) for key, value in metric_sums.items()}


def run_one_epoch(model: nn.Module, loader: DataLoader, optimizer, device: torch.device, config: Dict, epoch: int, scaler=None) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    metric_sums = {}
    total_examples = 0

    for batch in loader:
        batch = to_device(batch, device)
        batch_size = int(batch["activity"].shape[0])

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        use_amp = bool(config["use_amp"]) and device.type == "cuda"

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(batch["x"], batch["activity"])
                loss_dict = compute_loss(batch, outputs, config, epoch)

            if is_train:
                if scaler is not None and use_amp:
                    scaler.scale(loss_dict["total_loss"]).backward()
                    scaler.unscale_(optimizer)
                    if float(config["gradient_clip_norm"]) > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), float(config["gradient_clip_norm"]))
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss_dict["total_loss"].backward()
                    if float(config["gradient_clip_norm"]) > 0:
                        nn.utils.clip_grad_norm_(model.parameters(), float(config["gradient_clip_norm"]))
                    optimizer.step()

        aggregate(metric_sums, loss_dict, batch_size)
        total_examples += batch_size

    return finalize(metric_sums, total_examples)


def save_checkpoint(path: Path, model: nn.Module, optimizer, epoch: int, best_val_loss: float, config: Dict, activity_mapping: Dict, scaler_stats: Dict, history: List[Dict], model_summary: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": int(epoch),
            "best_val_loss": float(best_val_loss),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "activity_mapping": activity_mapping,
            "scaler_stats": scaler_stats,
            "training_history": history,
            "model_summary": model_summary,
        },
        path,
    )


def plot_training_history(history_df: pd.DataFrame, save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    for col in ["train_total_loss", "val_total_loss", "train_recon_loss", "val_recon_loss"]:
        if col in history_df.columns:
            ax.plot(history_df["epoch"], history_df[col], label=col)
    ax.set_title("Paper-style TimeVAE training history")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_reconstruction_examples(model: nn.Module, loader: DataLoader, device: torch.device, stats: Dict, paths: Dict[str, Path], config: Dict) -> None:
    model.eval()
    try:
        batch = next(iter(loader))
    except StopIteration:
        return

    batch = to_device(batch, device)
    with torch.no_grad():
        outputs = model(batch["x"], batch["activity"])

    real = inverse_normalize_fused(safe_numpy(batch["x"]), stats)
    recon = inverse_normalize_fused(safe_numpy(outputs["recon"]), stats)

    n_examples = min(int(config["num_reconstruction_examples"]), real.shape[0])
    columns = [(0, "ACC_x"), (3, "BVP"), (4, "EDA")]
    fig, axes = plt.subplots(n_examples, len(columns), figsize=(15, 3 * n_examples))
    if n_examples == 1:
        axes = np.expand_dims(axes, axis=0)

    for row in range(n_examples):
        for col, (channel_idx, channel_name) in enumerate(columns):
            ax = axes[row, col]
            ax.plot(real[row, :, channel_idx], label="real", linewidth=1)
            ax.plot(recon[row, :, channel_idx], label="recon", linewidth=1, alpha=0.8)
            ax.set_title(f"{channel_name} fused reconstruction example {row}")
            ax.grid(alpha=0.2)
            if row == 0 and col == 0:
                ax.legend()

    fig.tight_layout()
    save_path = paths["figures_dir"] / "timevae_fused_reconstruction_examples.png"
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def train_timevae(config: Dict = TIMEVAE_CONFIG) -> Dict[str, object]:
    set_random_seed(int(config["random_seed"]))
    paths = get_paths(config)
    make_dirs(paths["configs_dir"], paths["checkpoint_dir"], paths["generator_dir"], paths["results_dir"], paths["figures_dir"], paths["logs_dir"])

    logger = setup_logger(paths["logs_dir"] / "timevae_training.log")
    save_json(config, paths["configs_dir"] / "timevae_config.json")

    logger.info("Starting paper-style TimeVAE training")
    logger.info(f"Project root: {paths['root']}")
    logger.info(f"Real dir: {paths['real_dir']}")

    X_native, y_original, subjects = load_native_arrays(paths["real_dir"], config)
    validate_subject_split(subjects, config)
    split_indices = make_split_indices(subjects, config)

    logger.info("Building fused [N,512,6] representation")
    X_fused = native_to_fused512(X_native)

    activity_mapping = build_activity_mapping(config["activity_ids"])
    y_idx = original_labels_to_indices(y_original, activity_mapping)

    scaler_stats = compute_fused_scaler_stats(X_fused, split_indices["train"])
    X_fused_norm = normalize_fused(X_fused, scaler_stats)

    save_json(scaler_stats, paths["results_dir"] / "timevae_fused_scaler_stats.json")
    save_json(activity_mapping, paths["results_dir"] / "activity_mapping.json")
    np.savez(paths["results_dir"] / "split_indices_used_for_timevae.npz", train_idx=split_indices["train"], val_idx=split_indices["val"], test_idx=split_indices["test"])
    split_summary_df = save_split_summary(y_original, subjects, split_indices, paths, config)

    logger.info(f"Fused shape: {X_fused.shape}")
    logger.info(f"Train windows: {len(split_indices['train'])}")
    logger.info(f"Val windows: {len(split_indices['val'])}")
    logger.info(f"Test windows reserved: {len(split_indices['test'])}")

    train_loader, val_loader = make_dataloaders(X_fused_norm, y_idx, split_indices, config)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")

    model = PaperStyleTimeVAE(
        input_channels=int(config["input_channels"]),
        input_length=int(config["input_length"]),
        num_activities=len(config["activity_ids"]),
        latent_dim=int(config["latent_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        activity_embedding_dim=int(config["activity_embedding_dim"]),
        dropout=float(config["dropout"]),
    ).to(device)

    model_summary = {
        **count_parameters(model),
        "architecture": "paperstyle_fused_activity_conditioned_timevae",
        "internal_input_shape": ["N", 512, 6],
        "fused_channel_names": FUSED_CHANNEL_NAMES,
        "output_native_shapes": {"acc": ["N", 256, 3], "bvp": ["N", 512, 1], "slow": ["N", 32, 2]},
        "latent_dim": int(config["latent_dim"]),
        "hidden_dim": int(config["hidden_dim"]),
        "num_activities": len(config["activity_ids"]),
    }
    save_json(model_summary, paths["generator_dir"] / "timevae_model_summary.json")
    logger.info(f"Model summary: {model_summary}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config["learning_rate"]), weight_decay=float(config["weight_decay"]))
    use_amp = bool(config["use_amp"]) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val_loss = float("inf")
    best_epoch = -1
    epochs_without_improvement = 0
    history = []
    start_time = time.time()

    for epoch in range(1, int(config["max_epochs"]) + 1):
        train_metrics = run_one_epoch(model, train_loader, optimizer, device, config, epoch, scaler)
        val_metrics = run_one_epoch(model, val_loader, None, device, config, epoch, None)

        row = {"epoch": epoch}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)

        val_loss = float(val_metrics["total_loss"])
        logger.info(
            f"Epoch {epoch:03d} | train_total={train_metrics['total_loss']:.6f} | "
            f"val_total={val_metrics['total_loss']:.6f} | val_recon={val_metrics['recon_loss']:.6f} | "
            f"val_kl={val_metrics['kl_loss']:.6f} | val_act_acc={val_metrics['activity_accuracy']:.4f}"
        )

        save_checkpoint(paths["checkpoint_dir"] / "timevae_last.pt", model, optimizer, epoch, best_val_loss, config, activity_mapping, scaler_stats, history, model_summary)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            save_checkpoint(paths["checkpoint_dir"] / "timevae_best.pt", model, optimizer, epoch, best_val_loss, config, activity_mapping, scaler_stats, history, model_summary)
            logger.info(f"New best checkpoint saved at epoch {epoch}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= int(config["patience"]):
            logger.info(f"Early stopping at epoch {epoch}")
            break

    elapsed = time.time() - start_time
    history_df = pd.DataFrame(history)
    history_df.to_csv(paths["results_dir"] / "training_history.csv", index=False)

    final_metrics = {
        "best_epoch": int(best_epoch),
        "best_val_total_loss": float(best_val_loss),
        "elapsed_seconds": float(elapsed),
        "num_train_windows": int(len(split_indices["train"])),
        "num_val_windows": int(len(split_indices["val"])),
        "num_test_windows_reserved": int(len(split_indices["test"])),
        **model_summary,
    }
    save_json(final_metrics, paths["results_dir"] / "final_metrics.json")
    plot_training_history(history_df, paths["figures_dir"] / "training_loss_curve.png")

    if bool(config["save_reconstruction_examples"]):
        plot_reconstruction_examples(model, val_loader, device, scaler_stats, paths, config)

    logger.info("Training complete")
    logger.info(f"Best epoch: {best_epoch}")
    logger.info(f"Best val loss: {best_val_loss:.6f}")

    return {
        "model": model,
        "paths": paths,
        "history_df": history_df,
        "final_metrics": final_metrics,
        "split_summary_df": split_summary_df,
        "scaler_stats": scaler_stats,
        "activity_mapping": activity_mapping,
    }


# ============================================================
# Generation
# ============================================================

def load_timevae_checkpoint(checkpoint_path: Path, map_location: Optional[str] = None) -> Dict:
    checkpoint_path = require_file(checkpoint_path)
    if map_location is None:
        map_location = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        return torch.load(checkpoint_path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(checkpoint_path, map_location=map_location)


def build_model_from_checkpoint(checkpoint: Dict, device: torch.device) -> PaperStyleTimeVAE:
    config = checkpoint["config"]
    model = PaperStyleTimeVAE(
        input_channels=int(config["input_channels"]),
        input_length=int(config["input_length"]),
        num_activities=len(config["activity_ids"]),
        latent_dim=int(config["latent_dim"]),
        hidden_dim=int(config["hidden_dim"]),
        activity_embedding_dim=int(config["activity_embedding_dim"]),
        dropout=float(config["dropout"]),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def get_train_activity_distribution(y_original: np.ndarray, subjects: np.ndarray, config: Dict) -> Dict[str, object]:
    train_set = set(config["train_subjects"])
    keep = np.asarray([str(s) in train_set for s in subjects], dtype=bool)
    train_y = y_original[keep].astype(np.int64)

    counts = {int(activity): int(np.sum(train_y == int(activity))) for activity in config["activity_ids"]}
    total = int(sum(counts.values()))
    probabilities = {int(activity): float(counts[int(activity)] / total) for activity in config["activity_ids"]}
    return {"counts": counts, "probabilities": probabilities, "total": total}


def sample_activity_labels(distribution: Dict[str, object], config: Dict, rng: np.random.Generator, total_windows: int) -> np.ndarray:
    activity_ids = np.asarray(config["activity_ids"], dtype=np.int64)
    if config["activity_sampling_mode"] == "train_distribution":
        probabilities = np.asarray([distribution["probabilities"][int(a)] for a in activity_ids], dtype=np.float64)
        probabilities = probabilities / probabilities.sum()
        labels = rng.choice(activity_ids, size=total_windows, replace=True, p=probabilities)
    elif config["activity_sampling_mode"] == "uniform":
        labels = rng.choice(activity_ids, size=total_windows, replace=True)
    else:
        raise ValueError("activity_sampling_mode must be 'train_distribution' or 'uniform'.")
    return labels.astype(np.int64)


def build_synthetic_subjects(config: Dict) -> np.ndarray:
    subjects = []
    for subject_id in range(1, int(config["num_synthetic_subjects"]) + 1):
        label = f"synthetic_subject_{subject_id:02d}"
        subjects.extend([label] * int(config["windows_per_synthetic_subject"]))
    return np.asarray(subjects, dtype=object)


@torch.no_grad()
def generate_timevae_synthetic_subjects(config: Dict = TIMEVAE_CONFIG) -> Dict[str, object]:
    set_random_seed(int(config["random_seed"]))
    paths = get_paths(config)
    make_dirs(paths["synthetic_method_dir"], paths["generation_results_dir"], paths["generation_figures_dir"], paths["generation_results_base_dir"])

    checkpoint_path = paths["checkpoint_dir"] / "timevae_best.pt"
    checkpoint = load_timevae_checkpoint(checkpoint_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model_from_checkpoint(checkpoint, device)

    activity_mapping = checkpoint["activity_mapping"]
    scaler_stats = checkpoint["scaler_stats"]

    X_native_real, y_original, subjects_original = load_native_arrays(paths["real_dir"], config)
    activity_distribution = get_train_activity_distribution(y_original, subjects_original, config)

    total_windows = int(config["num_synthetic_subjects"]) * int(config["windows_per_synthetic_subject"])
    rng = np.random.default_rng(int(config["random_seed"]))

    y_generated_original = sample_activity_labels(activity_distribution, config, rng, total_windows)
    y_generated_idx = original_labels_to_indices(y_generated_original, activity_mapping)
    subjects_generated = build_synthetic_subjects(config)

    batch_size = int(config["generation_batch_size"])
    fused_batches = []

    for start in range(0, total_windows, batch_size):
        end = min(start + batch_size, total_windows)
        current_size = end - start
        activity_batch = torch.tensor(y_generated_idx[start:end], dtype=torch.long, device=device)
        z = torch.randn(current_size, int(config["latent_dim"]), device=device)
        fused_norm = safe_numpy(model.decode(z, activity_batch))
        fused = inverse_normalize_fused(fused_norm, scaler_stats)
        if bool(config["clip_generated_to_train_range"]):
            fused = clip_fused_to_train_range(fused, scaler_stats)
        fused_batches.append(fused.astype(np.float32))

    X_fused = np.concatenate(fused_batches, axis=0).astype(np.float32)
    X_native = fused512_to_native(X_fused)
    X_acc = X_native["acc"]
    X_bvp = X_native["bvp"]
    X_slow = X_native["slow"]

    synthetic_dir = paths["synthetic_method_dir"]
    np.save(synthetic_dir / "generated_subjects_X_acc_32hz.npy", X_acc)
    np.save(synthetic_dir / "generated_subjects_X_bvp_64hz.npy", X_bvp)
    np.save(synthetic_dir / "generated_subjects_X_slow_4hz.npy", X_slow)
    np.save(synthetic_dir / "generated_subjects_all_y.npy", y_generated_original.astype(np.int64))
    np.save(synthetic_dir / "generated_subjects_all_subject.npy", subjects_generated.astype(object))

    metadata_df = pd.DataFrame({
        "window_index": np.arange(total_windows, dtype=np.int64),
        "synthetic_subject": subjects_generated.astype(str),
        "activity_label": y_generated_original.astype(np.int64),
        "activity_index": y_generated_idx.astype(np.int64),
        "model_family": str(config["model_family"]),
        "generation_method": str(config["generation_method"]),
        "method_display_name": str(config["method_display_name"]),
        "internal_representation": "fused_512x6",
    })
    metadata_df.to_csv(synthetic_dir / "generated_subjects_metadata.csv", index=False)

    activity_counts = {str(int(a)): int(np.sum(y_generated_original == int(a))) for a in config["activity_ids"]}
    subject_counts = {str(s): int(np.sum(subjects_generated.astype(str) == str(s))) for s in sorted(np.unique(subjects_generated.astype(str)), key=subject_sort_key)}

    generation_summary = {
        "model_family": str(config["model_family"]),
        "method": str(config["generation_method"]),
        "method_display_name": str(config["method_display_name"]),
        "checkpoint": str(checkpoint_path),
        "internal_generation_shape": list(X_fused.shape),
        "num_generated_windows": int(total_windows),
        "num_synthetic_subjects": int(config["num_synthetic_subjects"]),
        "windows_per_subject": int(config["windows_per_synthetic_subject"]),
        "X_acc_shape": list(X_acc.shape),
        "X_bvp_shape": list(X_bvp.shape),
        "X_slow_shape": list(X_slow.shape),
        "activity_counts_original_labels": activity_counts,
        "subject_counts": subject_counts,
        "activity_sampling_mode": str(config["activity_sampling_mode"]),
        "uses_activity_conditioning": True,
        "synthetic_output_folder": str(synthetic_dir),
        "generation_parameters": {
            "latent_sampling": "standard_normal_prior",
            "internal_representation": "fused_512x6",
            "converted_to_native_rate_outputs": True,
            "clip_generated_to_train_range": bool(config["clip_generated_to_train_range"]),
        },
    }
    save_json(generation_summary, paths["generation_results_dir"] / "generation_summary.json")

    combined_generation_summary = {
        "model_family": str(config["model_family"]),
        "methods_run": [str(config["generation_method"])],
        "method_display_names": {str(config["generation_method"]): str(config["method_display_name"])},
        "synthetic_base": str(paths["root"] / config["synthetic_base_dir"]),
        "results_base": str(paths["root"] / config["generation_results_base_dir"]),
        "figures_base": str(paths["root"] / config["generation_figures_base_dir"]),
        "method_summaries": {str(config["generation_method"]): generation_summary},
    }
    save_json(combined_generation_summary, paths["generation_results_base_dir"] / "combined_generation_summary.json")

    plot_generated_examples(X_acc, X_bvp, X_slow, y_generated_original, paths, config)

    print("Saved synthetic data to:", synthetic_dir)
    print(json.dumps(generation_summary, indent=2))

    return {
        "X_fused": X_fused,
        "X_acc": X_acc,
        "X_bvp": X_bvp,
        "X_slow": X_slow,
        "y": y_generated_original,
        "subjects": subjects_generated,
        "metadata_df": metadata_df,
        "generation_summary": generation_summary,
        "paths": paths,
    }


def plot_generated_examples(X_acc: np.ndarray, X_bvp: np.ndarray, X_slow: np.ndarray, y: np.ndarray, paths: Dict[str, Path], config: Dict) -> None:
    rng = np.random.default_rng(int(config["random_seed"]))
    n = min(4, len(y))
    indices = rng.choice(np.arange(len(y)), size=n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(15, 3 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        axes[row, 0].plot(X_acc[idx, :, 0], linewidth=1)
        axes[row, 0].set_title(f"Generated ACC_x | activity {int(y[idx])}")
        axes[row, 1].plot(X_bvp[idx, :, 0], linewidth=1)
        axes[row, 1].set_title(f"Generated BVP | activity {int(y[idx])}")
        axes[row, 2].plot(X_slow[idx, :, 0], linewidth=1)
        axes[row, 2].set_title(f"Generated EDA | activity {int(y[idx])}")
        for col in range(3):
            axes[row, col].grid(alpha=0.2)

    fig.tight_layout()
    save_path = paths["generation_figures_dir"] / "timevae_generated_examples.png"
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Main
# ============================================================

def main(config: Dict = TIMEVAE_CONFIG) -> Dict[str, object]:
    training_outputs = train_timevae(config)
    generation_outputs = generate_timevae_synthetic_subjects(config)
    return {"training_outputs": training_outputs, "generation_outputs": generation_outputs}


if __name__ == "__main__":
    outputs = main(TIMEVAE_CONFIG)


2026-07-08 02:13:30,684 | INFO | Starting paper-style TimeVAE training
2026-07-08 02:13:30,685 | INFO | Project root: /home/iailab42/khans1/projects/ir
2026-07-08 02:13:30,685 | INFO | Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
2026-07-08 02:13:30,864 | INFO | Building fused [N,512,6] representation
2026-07-08 02:13:34,679 | INFO | Fused shape: (46925, 512, 6)
2026-07-08 02:13:34,679 | INFO | Train windows: 30762
2026-07-08 02:13:34,680 | INFO | Val windows: 6100
2026-07-08 02:13:34,680 | INFO | Test windows reserved: 10063
2026-07-08 02:13:34,822 | INFO | Device: cuda
2026-07-08 02:13:34,977 | INFO | Model summary: {'total_parameters': 2195854, 'trainable_parameters': 2195854, 'architecture': 'paperstyle_fused_activity_conditioned_timevae', 'internal_input_shape': ['N', 512, 6], 'fused_channel_names': ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP'], 'output_native_shapes': {'acc': ['N', 256, 3], 'bvp': ['N', 512, 1], 'slow': ['N', 32, 2]}, 'latent_dim': 

Saved synthetic data to: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/timevae/prior_v1
{
  "model_family": "timevae",
  "method": "prior_v1",
  "method_display_name": "TimeVAE-Prior",
  "checkpoint": "/home/iailab42/khans1/projects/ir/models/checkpoints/timevae_best.pt",
  "internal_generation_shape": [
    30000,
    512,
    6
  ],
  "num_generated_windows": 30000,
  "num_synthetic_subjects": 10,
  "windows_per_subject": 3000,
  "X_acc_shape": [
    30000,
    256,
    3
  ],
  "X_bvp_shape": [
    30000,
    512,
    1
  ],
  "X_slow_shape": [
    30000,
    32,
    2
  ],
  "activity_counts_original_labels": {
    "1": 2901,
    "2": 2137,
    "3": 1427,
    "4": 2268,
    "5": 4457,
    "6": 8587,
    "7": 2864,
    "8": 5359
  },
  "subject_counts": {
    "synthetic_subject_01": 3000,
    "synthetic_subject_02": 3000,
    "synthetic_subject_03": 3000,
    "synthetic_subject_04": 3000,
    "synthetic_subject_05": 3000,
    "synthetic_subject_06": 3000,
    "synthetic_

## Final run

This trains the improved paper-style TimeVAE and then generates synthetic subjects.

Recommended default:

```text
latent_dim = 128
hidden_dim = 128
reconstruction_weight = 3.5
beta_kl = 0.0001
kl_warmup_epochs = 30
```

After this finishes, use the model-family selector in notebooks 04, 05, and 06:

```python
SELECTED_MODEL_FAMILY = "timevae"
```


In [2]:
TIMEVAE_CONFIG["project_root"] = "/home/iailab42/khans1/projects/ir"

TIMEVAE_CONFIG["model_family"] = "timevae"
TIMEVAE_CONFIG["generation_method"] = "prior_v1"
TIMEVAE_CONFIG["method_display_name"] = "TimeVAE-Prior"

TIMEVAE_CONFIG["real_dir"] = "data/processed/native_rates"

TIMEVAE_CONFIG["max_epochs"] = 100
TIMEVAE_CONFIG["patience"] = 15
TIMEVAE_CONFIG["batch_size"] = 128

TIMEVAE_CONFIG["latent_dim"] = 128
TIMEVAE_CONFIG["hidden_dim"] = 128
TIMEVAE_CONFIG["activity_embedding_dim"] = 16

TIMEVAE_CONFIG["reconstruction_weight"] = 3.5
TIMEVAE_CONFIG["beta_kl"] = 0.0001
TIMEVAE_CONFIG["kl_warmup_epochs"] = 30
TIMEVAE_CONFIG["activity_loss_weight"] = 0.10

TIMEVAE_CONFIG["num_synthetic_subjects"] = 10
TIMEVAE_CONFIG["windows_per_synthetic_subject"] = 3000
TIMEVAE_CONFIG["activity_sampling_mode"] = "train_distribution"

TIMEVAE_CONFIG["clip_generated_to_train_range"] = False

outputs = main(TIMEVAE_CONFIG)
outputs["generation_outputs"]["generation_summary"]


2026-07-08 02:16:41,714 | INFO | Starting paper-style TimeVAE training
2026-07-08 02:16:41,715 | INFO | Project root: /home/iailab42/khans1/projects/ir
2026-07-08 02:16:41,715 | INFO | Real dir: /home/iailab42/khans1/projects/ir/data/processed/native_rates
2026-07-08 02:16:41,891 | INFO | Building fused [N,512,6] representation
2026-07-08 02:16:45,734 | INFO | Fused shape: (46925, 512, 6)
2026-07-08 02:16:45,734 | INFO | Train windows: 30762
2026-07-08 02:16:45,734 | INFO | Val windows: 6100
2026-07-08 02:16:45,735 | INFO | Test windows reserved: 10063
2026-07-08 02:16:45,874 | INFO | Device: cuda
2026-07-08 02:16:45,887 | INFO | Model summary: {'total_parameters': 2195854, 'trainable_parameters': 2195854, 'architecture': 'paperstyle_fused_activity_conditioned_timevae', 'internal_input_shape': ['N', 512, 6], 'fused_channel_names': ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP'], 'output_native_shapes': {'acc': ['N', 256, 3], 'bvp': ['N', 512, 1], 'slow': ['N', 32, 2]}, 'latent_dim': 

Saved synthetic data to: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/timevae/prior_v1
{
  "model_family": "timevae",
  "method": "prior_v1",
  "method_display_name": "TimeVAE-Prior",
  "checkpoint": "/home/iailab42/khans1/projects/ir/models/checkpoints/timevae_best.pt",
  "internal_generation_shape": [
    30000,
    512,
    6
  ],
  "num_generated_windows": 30000,
  "num_synthetic_subjects": 10,
  "windows_per_subject": 3000,
  "X_acc_shape": [
    30000,
    256,
    3
  ],
  "X_bvp_shape": [
    30000,
    512,
    1
  ],
  "X_slow_shape": [
    30000,
    32,
    2
  ],
  "activity_counts_original_labels": {
    "1": 2901,
    "2": 2137,
    "3": 1427,
    "4": 2268,
    "5": 4457,
    "6": 8587,
    "7": 2864,
    "8": 5359
  },
  "subject_counts": {
    "synthetic_subject_01": 3000,
    "synthetic_subject_02": 3000,
    "synthetic_subject_03": 3000,
    "synthetic_subject_04": 3000,
    "synthetic_subject_05": 3000,
    "synthetic_subject_06": 3000,
    "synthetic_

{'model_family': 'timevae',
 'method': 'prior_v1',
 'method_display_name': 'TimeVAE-Prior',
 'checkpoint': '/home/iailab42/khans1/projects/ir/models/checkpoints/timevae_best.pt',
 'internal_generation_shape': [30000, 512, 6],
 'num_generated_windows': 30000,
 'num_synthetic_subjects': 10,
 'windows_per_subject': 3000,
 'X_acc_shape': [30000, 256, 3],
 'X_bvp_shape': [30000, 512, 1],
 'X_slow_shape': [30000, 32, 2],
 'activity_counts_original_labels': {'1': 2901,
  '2': 2137,
  '3': 1427,
  '4': 2268,
  '5': 4457,
  '6': 8587,
  '7': 2864,
  '8': 5359},
 'subject_counts': {'synthetic_subject_01': 3000,
  'synthetic_subject_02': 3000,
  'synthetic_subject_03': 3000,
  'synthetic_subject_04': 3000,
  'synthetic_subject_05': 3000,
  'synthetic_subject_06': 3000,
  'synthetic_subject_07': 3000,
  'synthetic_subject_08': 3000,
  'synthetic_subject_09': 3000,
  'synthetic_subject_10': 3000},
 'activity_sampling_mode': 'train_distribution',
 'uses_activity_conditioning': True,
 'synthetic_outp